## Batch processing API to get path explanation

In [1]:
import anthropic
import json
import csv
import time
from pathlib import Path
import pandas as pd

client = anthropic.Anthropic()



In [ ]:
def prepare_batch_requests(csv_file_path, output_jsonl_path, start_index=0, end_index=100):
    """
    Convert CSV data into batch API JSONL format.
    Each line is a separate request for KG triplet extraction.
    
    Args:
        csv_file_path: Path to input CSV file
        output_jsonl_path: Path to output JSONL file
        start_index: Starting row index (0-based, inclusive)
        end_index: Ending row index (0-based, exclusive)
    """
    with open(csv_file_path, 'r', encoding='utf-8') as csv_file, \
         open(output_jsonl_path, 'w', encoding='utf-8') as jsonl_file:
        
        reader = csv.DictReader(csv_file)
        
        for idx, row in enumerate(reader):
            # Skip rows before start_index
            if idx < start_index:
                continue
            # Stop after end_index
            if idx >= end_index:
                break
                
            text_id = idx  # Adjust column name as needed
            text_triplets = row['document_triplets']  # Adjust column name as needed
            # summary_triplets = row['gen_summary_triplets']  # Adjust column name as needed
            summary_triplets = row['gen_summary_triplets']  # Adjust column name as needed
            
            # Create the batch request
            batch_request = {
                "custom_id": f"kg-extraction-{text_id}",
                "params": {
                    "model": "claude-sonnet-4-20250514",
                    # "model": "claude-sonnet-4-5-20250929",
                    "max_tokens": 20000,
                    'temperature': 0.1,
                    "system": "You are a helpful assistant in traversing and reasoning about knowledge graphs",
                    "messages": [
                        {
                            "role": "user",
                            "content": f"""
        Given detailed Text Triplets and Summary Triplets, identify supporting evidence paths that justify each summary triple. Think step-by-step and show your reasoning chain.

Algorithm:
(1) Create the knowledge graph using Text Triplets.
(2) For each summary triplet, identify the two entities and the relation between them.
(3) For each entity in the summary triplet, create a mapper for the possible entities in the knowledge graph.
(4) For each summary triplet, traverse the constructed knowledge graph and identify supporting paths. Use the previously created mapper to resolve the entities. Perform multi-hopping if needed.
(5) When there is no match, identify the Text Triplets where no match occurs. Record the triplets in the text where no match occurs.

Confidence Scale:
- 2 → Direct support: explicit edge or near-direct path exists in the KG
- 1 → Inferential support: implied through multi-hop traversal or categorical abstraction from multiple triplets
- 0 → Unsupported: no meaningful path found

Guidelines:
- If no Text Triplets support a Summary Triplet, set "is_supported": false and confidence:0
- In reasoning, explain either the connection OR why no connection exists
- Resolve the entities and relations if there are synonyms or variations
- Be strict: only mark as supported if there's clear evidence
- Consider summary triplets can be abstacted compared with the text triplets

Please only return a JSON with this structure: {{ "summary_triple": ["entity1", "relation", "entity2"], "is_supported": true/false, "supporting_evidence": [list of relevant Text Triplets] or [list of max 3 relevant but insufficient Text Triplets if unsupported], "confidence": 0-2 (only if supported, 0 otherwise), "reasoning": "Explain the explicit graph-based derivation OR clearly state which link is missing" }}

Text triplets = {text_triplets}
Summary Triplet = {summary_triplets}"""


                        }
                    ]
                }
            }
            
            jsonl_file.write(json.dumps(batch_request) + '\n')
    
    print(f"Batch requests prepared and saved to {output_jsonl_path}")

In [3]:
def submit_batch(jsonl_file_path):
    """
    Submit the batch of requests to Claude API.
    Returns the batch ID for tracking.
    """
    # Read all requests from the JSONL file
    requests = []
    with open(jsonl_file_path, 'r', encoding='utf-8') as f:
        for line in f:
            requests.append(json.loads(line))
    
    # Create the message batch
    message_batch = client.messages.batches.create(
        requests=requests
    )
    
    print(f"Batch submitted successfully!")
    print(f"Batch ID: {message_batch.id}")
    print(f"Status: {message_batch.processing_status}")
    
    return message_batch.id

def check_batch_status(batch_id):
    """
    Check the status of a batch processing job.
    """
    batch = client.messages.batches.retrieve(batch_id)
    
    print(f"\nBatch Status: {batch.processing_status}")
    print(f"Requests: {batch.request_counts}")
    
    return batch.processing_status

def retrieve_batch_results(batch_id, output_jsonl_path):
    """
    Retrieve results once batch processing is complete.
    """
    # Get all results
    results = client.messages.batches.results(batch_id)
    
    with open(output_jsonl_path, 'w', encoding='utf-8') as f:
        for result in results:
            f.write(json.dumps(result.model_dump()) + '\n')
    
    print(f"Results saved to {output_jsonl_path}")

def process_results(results_jsonl_path, output_csv_path):
    """
    Process the batch results and extract KG triplets into a CSV.
    """
    triplets = []
    
    with open(results_jsonl_path, 'r', encoding='utf-8') as f:
        for line in f:
            result = json.loads(line)
            
            # Extract custom_id to get original text ID
            custom_id = result['custom_id']
            text_id = custom_id.replace('kg-extraction-', '')
            
            # Check if request was successful
            if result['result']['type'] == 'succeeded':
                response = result['result']['message']
                
                # Extract the triplets from the response
                content = response['content'][0]['text']
                
                try:
                    # Parse the JSON response
                    extracted_triplets = json.loads(content)
                    
                    # Add text_id to each triplet
                    for triplet in extracted_triplets:
                        triplets.append({
                            'text_id': text_id,
                            'subject': triplet['subject'],
                            'predicate': triplet['predicate'],
                            'object': triplet['object']
                        })
                except json.JSONDecodeError:
                    print(f"Warning: Could not parse response for {custom_id}")
            else:
                print(f"Error processing {custom_id}: {result['result']}")
    
    # Write to CSV
    with open(output_csv_path, 'w', newline='', encoding='utf-8') as csvfile:
        fieldnames = ['text_id', 'subject', 'predicate', 'object']
        writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
        
        writer.writeheader()
        writer.writerows(triplets)
    
    print(f"Processed {len(triplets)} triplets and saved to {output_csv_path}")


In [6]:
def main():
    """
    Main workflow for batch KG extraction.
    """
    #     # File paths
    input_csv = "../data/test/data_with_triplets.csv"
    batch_requests_file = "../data/test/batch_requests_traversal_4.jsonl"
    batch_results_file = "../data/test/batch_results_traversal_4.jsonl"

    # Step 1: Prepare batch requests
    print("Step 1: Preparing batch requests...")
    prepare_batch_requests(input_csv, batch_requests_file)
    
    # Step 2: Submit batch
    print("\nStep 2: Submitting batch...")
    batch_id = submit_batch(batch_requests_file)
    
    # Step 3: Poll for completion
    print("\nStep 3: Waiting for batch to complete...")
    while True:
        status = check_batch_status(batch_id)
        
        if status == "ended":
            break
        
        print("Still processing... checking again in 60 seconds")
        time.sleep(60)
    
    # Step 4: Retrieve results
    print("\nStep 4: Retrieving results...")
    retrieve_batch_results(batch_id, batch_results_file)



In [7]:
if __name__ == "__main__":
    main()

Step 1: Preparing batch requests...
Batch requests prepared and saved to ../data/test/batch_requests_traversal_4.jsonl

Step 2: Submitting batch...
Batch submitted successfully!
Batch ID: msgbatch_01Rqkp9KbDP77PRF29MLeC7a
Status: in_progress

Step 3: Waiting for batch to complete...

Batch Status: in_progress
Requests: MessageBatchRequestCounts(canceled=0, errored=0, expired=0, processing=100, succeeded=0)
Still processing... checking again in 60 seconds

Batch Status: in_progress
Requests: MessageBatchRequestCounts(canceled=0, errored=0, expired=0, processing=100, succeeded=0)
Still processing... checking again in 60 seconds

Batch Status: in_progress
Requests: MessageBatchRequestCounts(canceled=0, errored=0, expired=0, processing=100, succeeded=0)
Still processing... checking again in 60 seconds

Batch Status: in_progress
Requests: MessageBatchRequestCounts(canceled=0, errored=0, expired=0, processing=100, succeeded=0)
Still processing... checking again in 60 seconds

Batch Status: i

### Format retrived output

In [ ]:
def extract_python_list_from_text(text_content):
    """
    Extract and parse a Python list from text content that contains markdown code blocks.
    
    Args:
        text_content: String containing text with a Python list in a code block
        
    Returns:
        Parsed Python list object, or None if extraction fails
    """
    import re
    import ast

    # Find content between ```json``` and ````
    pattern = r'```json\s*(.*?)\s*```'
    match = re.search(pattern, text_content, re.DOTALL)
    
    if match:
        list_str = match.group(1).strip()

        return list_str
        
        # Method 1: Try JSON parsing first (handles true/false/null)
        try:
            parsed_list = json.loads(list_str)
            return parsed_list
        except json.JSONDecodeError as e:
            # JSON failed, likely due to formatting issues
            pass
        
        # Method 2: Try ast.literal_eval (handles Python literals like True/False/None)
        try:
            # Replace JSON booleans/null with Python equivalents
            python_str = list_str.replace('true', 'True').replace('false', 'False').replace('null', 'None')
            parsed_list = ast.literal_eval(python_str)
            return parsed_list
        except (ValueError, SyntaxError) as e:
            # ast.literal_eval failed
            pass
        
        # Method 3: Manual JSON repair - fix common issues
        try:
            # Try to fix common JSON issues:
            # 1. Remove any trailing commas before closing braces/brackets
            fixed_str = re.sub(r',\s*([}\]])', r'\1', list_str)
            # 2. Ensure proper quotes (sometimes model uses smart quotes)
            fixed_str = fixed_str.replace('"', '"').replace('"', '"').replace("'", "'").replace("'", "'")
            
            parsed_list = json.loads(fixed_str)
            return parsed_list
        except json.JSONDecodeError:
            pass
        
        # Method 4: Last resort - use eval (unsafe but controlled environment)
        try:
            # Make true/false/null available for eval
            parsed_list = eval(list_str, {"__builtins__": {}, "true": True, "false": False, "null": None})
            return parsed_list
        except Exception as e:
            print(f"All parsing methods failed. Error: {e}")
            print(f"Content preview: {list_str[:200]}...")
            return None
    else:
        print("No <python> code block found in text")
        return None

In [9]:
def parse_results(batch_results_file, data_csv):

    """
    Parse batch results and align with original data CSV.
    
    Args:
        batch_results_file: Path to batch results JSONL file
        data_csv: Path to original data CSV file
    """
    llm_results = []

    df = pd.read_csv(data_csv)
    
    # Parse results
    with open(batch_results_file, 'r', encoding='utf-8') as f:
        for idx, line in enumerate(f):
            result = json.loads(line)
            id = result['custom_id'][14:]

            text_content = result['result']['message']['content'][0]['text']
            # print(text_content)

            extracted_list = extract_python_list_from_text(text_content)

            print(f"Extracted List for Text ID {id}: {extracted_list}")

            # text_id = df.iloc[idx]['id']
            llm_results.append({
                # 'id': text_id,  # Convert to int to match original CSV
                'fact_eval_res': extracted_list
            })

    return pd.DataFrame(llm_results)



In [10]:
batch_results_file = "../data/test/batch_results_traversal_4.jsonl"
data_csv_file = "../data/test/data_with_triplets.csv"

original_df = pd.read_csv(data_csv_file)

llm_results_df = parse_results(batch_results_file, data_csv_file)

# merge with original data if needed
# merged_df = pd.merge(original_df, llm_results_df, on='id', how='left')

original_df['sensitivity_fact_eval'] = llm_results_df['fact_eval_res']

Extracted List for Text ID 0: [
  {
    "summary_triple": ["More than 1,000 homeless people", "referred to", "charity"],
    "is_supported": true,
    "supporting_evidence": [["Prison Link Cymru", "had", "1,099 referrals"]],
    "confidence": 2,
    "reasoning": "Direct support exists. Prison Link Cymru (charity) had 1,099 referrals, which exceeds 1,000 and represents people seeking housing assistance (homeless people)."
  },
  {
    "summary_triple": ["charity", "located in", "Wales"],
    "is_supported": true,
    "supporting_evidence": [["Prison Link Cymru", "operates in", "2015-16"], ["Housing Act", "in", "Wales"]],
    "confidence": 1,
    "reasoning": "Inferential support through context. Prison Link Cymru operates within the Welsh housing system context, as evidenced by references to Welsh Housing Act and Welsh Government policies."
  },
  {
    "summary_triple": ["referral", "occurred in", "last year"],
    "is_supported": true,
    "supporting_evidence": [["Prison Link Cymru",

In [11]:
original_df.to_csv("../data/test/results_traversal_4.csv", index=False)

In [28]:
merged_df.head(10)

NameError: name 'merged_df' is not defined

In [72]:

# save merged results if needed
merged_df.to_csv("../data/test/results_traversal_3_fixed.csv", index=False)

In [73]:
og_file = pd.read_csv("../data/test/results_traversal_3.csv")
fix_file = pd.read_csv("../data/test/results_traversal_3_fixed.csv")

# Much faster - use merge to update values
# og_file_copy = og_file.copy()
# update_dict = fix_file.set_index('id')['fact_eval_res'].to_dict()
# og_file_copy['fact_eval_res'] = og_file_copy['id'].map(update_dict).fillna(og_file_copy['fact_eval_res'])

og_file_copy = og_file.copy()

# # Fix using .loc instead of chained indexing to avoid SettingWithCopyWarning
for i in range(len(og_file)):
    for j in range(len(fix_file)):
        if og_file.iloc[i]['id'] == fix_file.iloc[j]['id']:
            og_file_copy.loc[i, 'fact_eval_res'] = fix_file.iloc[j]['fact_eval_res']


In [74]:
len(og_file_copy)

100

In [75]:
og_file_copy.head(20)

,id,document,generated_summary,summary,document_triplets,gen_summary_triplets,fact_eval_res
0,38264402,"Prison Link Cymru had 1,099 referrals in 2015-...","More than 1,000 homeless people were referred ...","There is a ""chronic"" need for more housing for...","[['Prison Link Cymru', 'had', '1,099 referrals...","[['More than 1,000 homeless people', 'referred...","[\n{\n ""summary_triple"": [""More than 1,000 ho..."
1,34227252,Officers searched properties in the Waterfront...,A man has been charged after firearms and mone...,"A man has appeared in court after firearms, am...","[['Officers', 'searched', 'properties'], ['pro...","[['man', 'charged after', 'seizure'], ['police...","[\n{\n ""summary_triple"": [""man"", ""charged aft..."
2,38537698,"Jordan Hill, Brittany Covington and Tesfaye Co...",Four black people accused of torturing a menta...,Four people accused of kidnapping and torturin...,"[['Jordan Hill', 'age', '18'], ['Brittany Covi...","[['Four black people', 'accused of', 'torturin...","[\n{\n ""summary_triple"": [""Four black people""..."
3,36175342,The 48-year-old former Arsenal goalkeeper play...,West Bromwich Albion have appointed Reading di...,West Brom have appointed Nicky Hammond as tech...,"[['48-year-old former Arsenal goalkeeper', 'pl...","[['West Bromwich Albion', 'appointed', 'Nick H...","[\n {\n ""summary_triple"": [""West Bromwich ..."
4,39070183,Restoring the function of the organ - which he...,"The ""fasting-mimicking"" diet can regenerate th...",The pancreas can be triggered to regenerate it...,"[['organ', 'helps control', 'blood sugar level...","[['fasting-mimicking diet', 'can regenerate', ...","[\n {\n ""summary_triple"": [""fasting-mimick..."
5,38899892,But there certainly should be.\nThese are two ...,There is not a lot of buzz about the planned m...,Since their impending merger was announced in ...,"[['Essilor', 'is', ""France's company""], ['Essi...","[['Essilor', 'merging with', 'Luxottica'], ['E...","[\n {\n ""summary_triple"": [""Essilor"", ""mer..."
6,39339718,Media playback is not supported on this device...,naghel Houvenaghel has become the latest high-...,"A ""medal at any cost"" approach created a ""cult...","[['Wendy Houvenaghel', 'accused', 'British Cyc...","[['Naghel Houvenaghel', 'is', 'cyclist'], ['Na...","[\n{\n ""summary_triple"": [""Naghel Houvenaghel..."
7,34571446,It's no joke. But Kareem Badr says people did ...,"If you go to a comedy club in Texas, you're li...",Have you heard the one about the computer prog...,"[['Kareem Badr', 'co-founded', 'Hideout'], ['K...","[['you', 'go to', 'comedy club'], ['comedy clu...","[\n{\n ""summary_triple"": [""you"", ""go to"", ""co..."
8,36892983,Relieved that the giant telecoms company would...,"The news that Ofcom, the competition watchdog,...",The reaction from BT's investors told us much ...,"[['giant telecoms company', 'would not be', 'b...","[['Ofcom', 'is', 'competition watchdog'], ['Of...","[\n {\n ""summary_triple"": [""Ofcom"", ""is"", ..."
9,37732028,"""I'm really looking forward to it - the home o...",Celtic manager Brendan Rodgers is relishing th...,Manager Brendan Rodgers is sure Celtic can exp...,"[['Rodgers', 'is coach of', 'Celtic'], ['Rodge...","[['Brendan Rodgers', 'is', 'Celtic manager'], ...","[\n {\n ""summary_triple"": [""Brendan Rodger..."


In [76]:
og_file_copy.to_csv("../data/test/results_traversal_3_final.csv", index=False)